# FinTrust Financial Intelligence & Digital Banking Solution (Week 2)
## Data Analysis, Exploratory Visualizations, Baseline Modeling & Assistant Evaluation

This notebook contains complete execution steps for:
1. **Data Cleaning & Quality Audit**: Customer (1,500 rows) and Transaction (12,000 rows) datasets.
2. **Feature Engineering**: 8 predictive risk features created.
3. **Exploratory Data Analysis**: Financial volume, channel friction, cross-border risk, and hourly patterns.
4. **Machine Learning Baselines**: Logistic Regression & Random Forest evaluation with ROC-AUC & Confusion Matrices.
5. **GenAI Assistant Audit**: Grounding evaluation across 15 boundary test prompts.

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix, roc_curve

%matplotlib inline
plt.style.use('dark_background')

In [2]:
# Load Processed Datasets
df_cust = pd.read_csv('../data/processed/clean_customers.csv')
df_tx = pd.read_csv('../data/processed/clean_transactions.csv')
df_modelling = pd.read_csv('../data/processed/modelling_features.csv')

print(f'Clean Customers Shape: {df_cust.shape}')
print(f'Clean Transactions Shape: {df_tx.shape}')
print(f'Modelling Features Shape: {df_modelling.shape}')

In [3]:
# EDA Plot: Transaction Amount Distribution
plt.figure(figsize=(10, 5))
sns.histplot(df_modelling['Amount_NGN'], kde=True, color='#38bdf8', bins=30)
plt.title('Transaction Amount (NGN) Distribution', fontsize=14, color='white')
plt.xlabel('Amount NGN')
plt.ylabel('Frequency')
plt.show()

In [4]:
# Channel Friction & Failure Rate Analysis
channel_df = df_modelling.groupby('Channel').agg(
    Total_Volume=('Amount_NGN', 'sum'),
    Tx_Count=('Transaction_ID', 'count'),
    Failed_Count=('Transaction_Status', lambda x: (x == 'Failed').sum())
)
channel_df['Failure_Rate_%'] = (channel_df['Failed_Count'] / channel_df['Tx_Count']) * 100
display(channel_df)

In [5]:
# Baseline ML Classification
y = (df_modelling['Risk_Review_Flag'] == 'Yes').astype(int)
drop_cols = ['Risk_Review_Flag', 'Transaction_ID', 'Customer_ID', 'Customer_Name', 'Transaction_DateTime', 'Account_Status']
X = df_modelling.drop(columns=drop_cols)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
print(f'X_train shape: {X_train.shape}, X_test shape: {X_test.shape}')